This notebook tests basic simulation by recapitulating a simulated dataset several times and measuring variance.

Imports

In [1]:
import sys
import os
package_path = os.path.abspath("..")
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

2025-05-28 19:29:06.760472: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-28 19:29:06.765305: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-05-28 19:29:06.765318: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
#dask imports
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

Make the dask cluster & client in accordance with resource avail and model size

In [3]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=2:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)

Ship the package to the workers:

(This is again a dev hack which will not be required after code is packaged)

In [4]:
cluster.scale(jobs=5)

In [5]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [9]:
#let's make an ortho object out of the simulated data
test=scm.ortho()
test.criss_cross(client=client,dat=dat)
test.extract_params(client)

In [10]:
description_primordial=scm.describe_parameters(client,parameters=test.by_cre_parameters.result(),dat=dat,split="cre_id")

In [11]:
description_primordial=scm.auto_partition(description_primordial,50)

In [12]:
description_primordial.compute()

,C(cell_type)[blood],C(cell_type)[brain],C(rep_id)[1],C(rep_id)[2],C(rep_id)[3],cre_id,cells,nb,zi,theta,r,sigmasquare,p
0,1,0,0,1,0,nobody,503,1.769080,0.454869,0.771285,2.162544,3.216286,0.550038
1,1,0,0,0,1,nobody,500,1.769080,0.880428,0.771285,2.162544,3.216286,0.550038
2,1,0,1,0,0,nobody,499,1.769080,0.772933,0.771285,2.162544,3.216286,0.550038
3,0,1,0,1,0,nobody,443,0.927331,0.454869,0.771285,2.162544,1.324984,0.699881
4,0,1,0,0,1,nobody,441,0.927331,0.880428,0.771285,2.162544,1.324984,0.699881
5,0,1,1,0,0,nobody,440,0.927331,0.772933,0.771285,2.162544,1.324984,0.699881
6,1,0,0,1,0,somebody,522,12.467969,0.519615,1.110920,3.037152,63.650866,0.195881
7,1,0,1,0,0,somebody,504,12.467969,0.799793,1.110920,3.037152,63.650866,0.195881
8,1,0,0,0,1,somebody,498,12.467969,0.902763,1.110920,3.037152,63.650866,0.195881
9,0,1,1,0,0,somebody,464,9.960114,0.799793,1.110920,3.037152,42.623566,0.233676


In [27]:
x=scm.simulate_from_description(description_primordial)
x=x.compute()
sim=x.copy()
x=scm.undo_one_hot_encoding(x)
x=x.rename({'zinb_sample':'umis_mpra_bc'},axis=1)[['rep_id','cre_id','cell_type','umis_mpra_bc']]
x

,rep_id,cre_id,cell_type,umis_mpra_bc
2,2,nobody,blood,0
3,2,nobody,blood,0
7,2,nobody,blood,1
12,2,nobody,blood,0
13,2,nobody,blood,0
...,...,...,...,...
7084,3,neurogene,brain,0
7085,3,neurogene,brain,55
7089,3,neurogene,brain,0
7091,3,neurogene,brain,0


In [28]:
recap=scm.ortho()
recap.criss_cross(client=client,dat=x)
recap.extract_params(client)

In [29]:
description_x=scm.describe_parameters(client,parameters=recap.by_cre_parameters.result(),dat=x,split="cre_id")

In [ ]:
working=scm.undo_one_hot_encoding(description_x.reset_index())


#working.drop(columns=['cre_id','cells','nb','theta','p','r','sigmasquare']).drop_duplicates()#.groupby("rep_id").agg('mean')# zi
#working.drop(columns=['rep_id','cells','zi']).drop_duplicates()# nb

,zi,cell_type,rep_id
0,0.444624,blood,2
1,0.884219,blood,3
2,0.770853,blood,1
3,0.444624,brain,2
4,0.884219,brain,3
5,0.770853,brain,1
6,0.494735,blood,2
7,0.796856,blood,1
8,0.881731,blood,3
9,0.796856,brain,1


In [135]:
cluster.close()